# Map LA's air quality in an afternoon

**Terrain starter project · Python · EPA AirNow**

Pull air-quality readings for the Los Angeles area and map monitoring sites color-coded by AQI.

**You will build:** a map of LA-area stations with AQI-colored markers and a short written takeaway.

**Before you start**
- Works in [Google Colab](https://colab.research.google.com) or any Python environment.
- Live data needs a free [AirNow API key](https://docs.airnowapi.org/account/request/) (optional — sample data runs without one).
- Read the [EPA AirNow explainer](../explainer.html?id=epa-airnow) on Terrain for context.

## 1. Setup

In [ ]:
# Run once if packages are missing (Colab usually has these)
try:
    import folium
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'folium', '-q'])
    import folium

import json
from pathlib import Path
import pandas as pd
import requests
import matplotlib.pyplot as plt

## 2. Load air-quality observations

In [ ]:
USE_LIVE_DATA = False  # set True after you paste an API key below
AIRNOW_API_KEY = ""      # or: import os; AIRNOW_API_KEY = os.environ.get("AIRNOW_API_KEY", "")

SAMPLE_PATH = Path("data/airnow_la_sample.json")
if not SAMPLE_PATH.exists():
    SAMPLE_PATH = Path("notebooks/data/airnow_la_sample.json")

def fetch_airnow_live(api_key, lat=34.05, lon=-118.25, distance=75):
    url = "https://www.airnowapi.org/aq/observation/latLong/current/"
    params = {
        "format": "application/json",
        "latitude": lat,
        "longitude": lon,
        "distance": distance,
        "API_KEY": api_key,
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

if USE_LIVE_DATA and AIRNOW_API_KEY:
    records = fetch_airnow_live(AIRNOW_API_KEY)
    print(f"Fetched {len(records)} live observations from AirNow")
else:
    with open(SAMPLE_PATH) as f:
        records = json.load(f)
    print(f"Using bundled sample data ({len(records)} observations)")
    print("Tip: set USE_LIVE_DATA = True and add your AirNow API key for live readings.")

df = pd.DataFrame(records)
df.head()

## 3. Clean and summarize

In [ ]:
# Keep one row per site (highest AQI if duplicates)
clean = (
    df.assign(AQI=pd.to_numeric(df["AQI"], errors="coerce"))
      .dropna(subset=["Latitude", "Longitude", "AQI"])
      .sort_values("AQI", ascending=False)
      .drop_duplicates(subset=["SiteName"], keep="first")
      .rename(columns={"Latitude": "lat", "Longitude": "lon"})
)

print(f"Monitoring sites: {len(clean)}")
print(f"Median AQI: {clean['AQI'].median():.0f}")
print(f"Highest AQI: {clean['AQI'].max():.0f} at {clean.iloc[0]['SiteName']}")
clean[["SiteName", "ReportingArea", "ParameterName", "AQI", "Category"]].head(8)

## 4. Color stations by AQI

In [ ]:
def aqi_color(aqi):
    if aqi <= 50:
        return "#2F6B4F"   # Good
    if aqi <= 100:
        return "#D98E5A"   # Moderate
    if aqi <= 150:
        return "#C45C5C"   # Unhealthy for sensitive groups
    return "#7B2D26"       # Unhealthy+

clean = clean.assign(color=clean["AQI"].map(aqi_color))

## 5. Build the map

In [ ]:
m = folium.Map(location=[34.05, -118.25], zoom_start=9, tiles="CartoDB positron")

for _, row in clean.iterrows():
    folium.CircleMarker(
        location=[row.lat, row.lon],
        radius=9,
        color=row.color,
        fill=True,
        fill_color=row.color,
        fill_opacity=0.85,
        popup=f"<b>{row.SiteName}</b><br>AQI {int(row.AQI)} ({row.Category})<br>{row.ParameterName}",
    ).add_to(m)

m

## 6. Static chart (optional export)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(clean["SiteName"], clean["AQI"], color=clean["color"])
ax.set_xlabel("AQI")
ax.set_title("Current AQI by monitoring site (LA region)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Write your takeaway

Answer in 3–4 sentences:

1. Which areas show the highest AQI in this snapshot?
2. Is ozone or PM2.5 driving the peak readings?
3. What limitation should readers keep in mind about monitor spacing?

**Next:** Compare these readings to long-term burden scores in the [CalEnviroScreen project](calenviroscreen_burden.Rmd) or browse more datasets on [Terrain](../datasets.html).